# Evaluate Human + AI Proposals with `eval_ncems_criteria`

This notebook implements the evaluation step from the analysis plan:
- Use the prompt `eval_ncems_criteria` in `src/prompt_templates.py`
- For each proposal (human Y1, human Y2, baseline AI), call each evaluator model (GPT, Gemini, Claude) **one proposal per API call**
- Save all evaluations to a single JSON file in `data/reviews/ai_reviews/` with fields: `title`, `author`, `evaluator`, `evaluations` (JSON)

**Inputs**
- Human proposals: `data/human-proposals/human-proposals-y1.json`, `data/human-proposals/human-proposals-y2.json`
- Baseline AI proposals: `data/ai-proposals/baseline/ai_proposals_baseline_complete_20260209_205423.csv`

**Output**
- `data/reviews/ai_reviews/ai_reviews_ncems_criteria_<timestamp>.json`

## Setup

Install requirements (safe to skip if your environment already has them).

In [ ]:
# If needed, uncomment:
# %pip install -r src/requirements.txt

## Imports and configuration

In [16]:
import os
import sys
import json
import re
import hashlib
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import pandas as pd
from tqdm import tqdm
from dotenv import load_dotenv

# Ensure the notebook uses the local .env you edited (and overrides any stale kernel env vars)
load_dotenv(dotenv_path=Path('.env'), override=True)

# Add src to path to import custom modules
src_path = os.path.join(os.getcwd(), 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from ai_models_interface import AIModelsInterface
from prompt_templates import PromptManager


def _key_fingerprint(val: Optional[str]) -> str:
    if not val:
        return 'missing'
    return hashlib.sha256(val.encode('utf-8')).hexdigest()[:10]


print(f"✓ Working directory: {os.getcwd()}")
print(f"✓ Python path includes: {src_path}")
print(f"✓ GOOGLE_API_KEY fingerprint: {_key_fingerprint(os.getenv('GOOGLE_API_KEY'))}")
print(f"✓ OPENAI_API_KEY fingerprint: {_key_fingerprint(os.getenv('OPENAI_API_KEY'))}")
print(f"✓ ANTHROPIC_API_KEY fingerprint: {_key_fingerprint(os.getenv('ANTHROPIC_API_KEY'))}")

✓ Working directory: /Users/eveyhuang/Documents/NICO/human-AI-proposal
✓ Python path includes: /Users/eveyhuang/Documents/NICO/human-AI-proposal/src
✓ GOOGLE_API_KEY fingerprint: d8c05ee403
✓ OPENAI_API_KEY fingerprint: 2699fb9ec8
✓ ANTHROPIC_API_KEY fingerprint: c3907ca189


## Load call text (used in the evaluation prompt)

In [2]:
with open('data/call_and_info.json', 'r') as f:
    call_and_info = json.load(f)

research_call = call_and_info['call']

print("✓ Loaded research call")
print(f"Preview: {research_call[:200]}...")

✓ Loaded research call
Preview: This funding organization is dedicated to catalyzing multidisciplinary scientific teams to synthesize publicly available data to address fundamental questions related to emergence phenomena in the mol...


## Load proposals (human Y1/Y2 + baseline AI CSV)

In [3]:
# Inputs specified by the analysis plan
human_y1_path = Path('data/human-proposals/human-proposals-y1.json')
human_y2_path = Path('data/human-proposals/human-proposals-y2.json')
ai_baseline_csv = Path('data/ai-proposals/baseline/ai_proposals_baseline_complete_20260209_205423.csv')

if not human_y1_path.exists() or not human_y2_path.exists():
    raise FileNotFoundError("Missing human proposal files in data/human-proposals/")
if not ai_baseline_csv.exists():
    raise FileNotFoundError(f"Missing baseline AI CSV: {ai_baseline_csv}")

# Load human proposals
human_rows: List[Dict[str, Any]] = []
for p in [human_y1_path, human_y2_path]:
    data = json.loads(p.read_text(encoding='utf-8'))
    proposals = data['proposals'] if isinstance(data, dict) and 'proposals' in data else data
    for pr in proposals:
        pr = dict(pr)
        pr['source_file'] = p.name
        human_rows.append(pr)

human_df = pd.DataFrame(human_rows)

# Load baseline AI proposals
ai_df = pd.read_csv(ai_baseline_csv)

print("✓ Loaded proposals")
print(f"  Human proposals: {len(human_df)} (files: {sorted(human_df['source_file'].unique().tolist())})")
print(f"  AI proposals: {len(ai_df)} (models: {ai_df['model'].value_counts().to_dict()})")
print(f"  Baseline AI CSV: {ai_baseline_csv.name}")

✓ Loaded proposals
  Human proposals: 23 (files: ['human-proposals-y1.json', 'human-proposals-y2.json'])
  AI proposals: 69 (models: {'gpt-5.2': 23, 'gemini-3-pro-preview': 23, 'claude-opus-4-5': 23})
  Baseline AI CSV: ai_proposals_baseline_complete_20260209_205423.csv


## Normalize proposal fields and create blinded IDs

We keep `author` metadata for saving, but do **not** include it in the evaluation prompt.

In [4]:
def _infer_human_author(source_file: str) -> str:
    # required output field: author = human-y1, human-y2, or which AI model
    sf = (source_file or '').lower()
    if 'y1' in sf:
        return 'human-y1'
    if 'y2' in sf:
        return 'human-y2'
    return 'human'


def create_full_text_ai(row: Dict[str, Any]) -> str:
    # Full proposal text (excluding title/abstract which are passed separately to the prompt)
    sections = [
        f"Background and significance: {row.get('background_and_significance', '')}",
        f"Research questions and hypotheses: {row.get('research_questions_and_hypotheses', '')}",
        f"Methods and approach: {row.get('methods_and_approach', '')}",
        f"Expected outcomes and impact: {row.get('expected_outcomes_and_impact', '')}",
        f"Open science and reproducibility: {row.get('open_science_and_reproducibility', '')}",
        f"Budget and resources: {row.get('budget_and_resources', '')}",
    ]
    return "\n\n".join([s for s in sections if s.split(': ', 1)[1].strip()])


def create_full_text_human(row: Dict[str, Any]) -> str:
    # Use the provided full draft as the "full proposal" field
    return (
        row.get('full_draft')
        or row.get('full_text')
        or row.get('full_proposal')
        or ''
    )


def blinded_proposal_id(title: str, abstract: str, full_text: str) -> str:
    # stable, content-derived ID that does not reveal authorship
    payload = (title or '') + "\n" + (abstract or '') + "\n" + (full_text or '')
    h = hashlib.sha1(payload.encode('utf-8')).hexdigest()[:10]
    return f"P{h}"


proposals: List[Dict[str, Any]] = []

# Human proposals
for _, r in human_df.iterrows():
    title = r.get('proposal_title') or r.get('title') or ''
    abstract = r.get('abstract') or ''
    full_text = create_full_text_human(r.to_dict())
    proposals.append(
        {
            'proposal_id': blinded_proposal_id(title, abstract, full_text),
            'title': title,
            'abstract': abstract,
            'full_text': full_text,
            'author': _infer_human_author(r.get('source_file', '')),
            'source': 'human',
        }
    )

# Baseline AI proposals
for _, r in ai_df.iterrows():
    title = r.get('title') or ''
    abstract = r.get('abstract') or ''
    full_text = create_full_text_ai(r.to_dict())
    proposals.append(
        {
            'proposal_id': blinded_proposal_id(title, abstract, full_text),
            'title': title,
            'abstract': abstract,
            'full_text': full_text,
            'author': str(r.get('model', 'ai')),
            'source': 'ai',
        }
    )

proposals_df = pd.DataFrame(proposals)

print("✓ Normalized proposal list")
print(f"  Total proposals: {len(proposals_df)}")
print(f"  By source: {proposals_df['source'].value_counts().to_dict()}")
print(f"  Example blinded IDs: {proposals_df['proposal_id'].head(3).tolist()}")

✓ Normalized proposal list
  Total proposals: 92
  By source: {'ai': 69, 'human': 23}
  Example blinded IDs: ['P1137f1830a', 'Pcc4e4d7ee4', 'P832af59140']


## Initialize models + prompt manager

In [18]:
ai_interface = AIModelsInterface(config_path='.env')
available_models = ai_interface.get_available_models()

prompt_manager = PromptManager()
_ = prompt_manager.get_template('eval_ncems_criteria')  # validate template exists

# Evaluators specified in the plan
requested_models = ['gpt-5.2', 'gemini-3-pro-preview', 'claude-opus-4-5']
candidate_models = [m for m in requested_models if m in available_models]

print(f"✓ Available models: {available_models}")
print(f"✓ Candidate evaluator models: {candidate_models}")


def _extract_first_json_object(text: str) -> Tuple[Optional[Dict[str, Any]], Optional[str]]:
    if text is None:
        return None, 'empty response'

    s = text.strip()
    if not s:
        return None, 'empty response'

    if s.startswith('{') and s.endswith('}'):
        try:
            return json.loads(s), None
        except Exception as e:
            return None, f'json parse error (full string): {e}'

    start = s.find('{')
    if start == -1:
        return None, 'no json object found'

    depth = 0
    end = None
    for i, ch in enumerate(s[start:], start=start):
        if ch == '{':
            depth += 1
        elif ch == '}':
            depth -= 1
            if depth == 0:
                end = i
                break

    if end is None:
        return None, 'unterminated json object'

    cand = s[start : end + 1]
    try:
        return json.loads(cand), None
    except Exception as e:
        return None, f'json parse error (extracted object): {e}'


def _preflight_model(model_name: str) -> Tuple[bool, str]:
    """Return (ok, message)."""
    probe_prompt = 'Return ONLY valid JSON: {"ok": true}'
    try:
        out = ai_interface.generate_content(probe_prompt, model_name=model_name, temperature=0)
    except Exception as e:
        return False, f"exception: {e}"

    if isinstance(out, str) and out.strip().startswith('Error:'):
        return False, out.strip()

    parsed, err = _extract_first_json_object(out if isinstance(out, str) else str(out))
    if parsed is None:
        return False, f"preflight returned non-JSON: {err}"

    return True, 'ok'


models_to_use: List[str] = []
model_failures: Dict[str, str] = {}

for m in candidate_models:
    ok, msg = _preflight_model(m)
    if ok:
        models_to_use.append(m)
        print(f"✓ Preflight OK: {m}")
    else:
        model_failures[m] = msg
        print(f"✗ Preflight FAILED: {m}\n  {msg}")

if not models_to_use:
    raise RuntimeError(
        "No evaluator models passed preflight. "
        "Fix your API keys in `.env` (e.g., renew `GOOGLE_API_KEY`, or set `OPENAI_API_KEY` and `ANTHROPIC_API_KEY`) "
        f"and retry. Candidate models were: {candidate_models}"
    )

print(f"\n✓ Using evaluator models: {models_to_use}")

INFO:ai_models_interface:OpenAI GPT-4 initialized
INFO:ai_models_interface:Google Gemini initialized
INFO:ai_models_interface:Anthropic Claude initialized


✓ Available models: ['gpt-5.2', 'gemini-3-pro-preview', 'claude-opus-4-5']
✓ Candidate evaluator models: ['gpt-5.2', 'gemini-3-pro-preview', 'claude-opus-4-5']


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:root:AFC is enabled with max remote calls: 10.


✓ Preflight OK: gpt-5.2


INFO:root:AFC remote call 1 is done.


✓ Preflight OK: gemini-3-pro-preview


INFO:httpx:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"


✓ Preflight OK: claude-opus-4-5

✓ Using evaluator models: ['gpt-5.2', 'gemini-3-pro-preview', 'claude-opus-4-5']


## Helpers: robust JSON extraction + prompt safety

In [19]:
def extract_first_json_object(text: str) -> Tuple[Optional[Dict[str, Any]], Optional[str]]:
    """Return (parsed_json, error_message)."""
    if text is None:
        return None, 'empty response'

    s = text.strip()
    if not s:
        return None, 'empty response'

    # Fast path
    if s.startswith('{') and s.endswith('}'):
        try:
            return json.loads(s), None
        except Exception as e:
            return None, f'json parse error (full string): {e}'

    # Extract first balanced {...}
    start = s.find('{')
    if start == -1:
        return None, 'no json object found'

    depth = 0
    end = None
    for i, ch in enumerate(s[start:], start=start):
        if ch == '{':
            depth += 1
        elif ch == '}':
            depth -= 1
            if depth == 0:
                end = i
                break

    if end is None:
        return None, 'unterminated json object'

    candidate = s[start : end + 1]
    try:
        return json.loads(candidate), None
    except Exception as e:
        return None, f'json parse error (extracted object): {e}'


def truncate_chars(text: str, max_chars: int) -> str:
    if text is None:
        return ''
    if len(text) <= max_chars:
        return text
    return text[:max_chars] + f"\n\n[TRUNCATED to first {max_chars} chars]"

## Run evaluations and save a single consolidated JSON (with resume)

This will create one API call per (proposal × evaluator).

In [20]:
output_dir = Path('data/reviews/ai_reviews')
output_dir.mkdir(parents=True, exist_ok=True)

run_ts = datetime.now().strftime('%Y%m%d_%H%M%S')
output_path = output_dir / f"ai_reviews_ncems_criteria_{run_ts}.json"

# Optional: set this to an existing JSON in output_dir to resume
resume_from: Optional[Path] = None

# Safety: truncate extremely long full texts to reduce context overflows
# (tune if needed; truncation note is included in the prompt text)
MAX_FULL_TEXT_CHARS = 60000


def make_prompt(proposal: Dict[str, Any]) -> str:
    # Do NOT include proposal['author'] in the prompt (blinding to authorship)
    data = {
        'research_call': research_call,
        'proposal_id': proposal['proposal_id'],
        'proposal_title': proposal['title'],
        'proposal_abstract': truncate_chars(proposal.get('abstract', ''), 4000),
        'proposal_full': truncate_chars(proposal.get('full_text', ''), MAX_FULL_TEXT_CHARS),
    }
    return prompt_manager.format_prompt('eval_ncems_criteria', data)


def call_evaluator(prompt: str, evaluator_model: str) -> str:
    # Model-specific kwargs for stability
    if evaluator_model.startswith('claude'):
        return ai_interface.generate_content(prompt, model_name=evaluator_model, temperature=0, max_tokens=8192)
    if evaluator_model.startswith('gpt'):
        return ai_interface.generate_content(prompt, model_name=evaluator_model, temperature=0, max_completion_tokens=4000)
    return ai_interface.generate_content(prompt, model_name=evaluator_model, temperature=0)


# Load existing results if resuming
reviews: List[Dict[str, Any]] = []
completed_keys: set[Tuple[str, str]] = set()

if resume_from is not None:
    if not resume_from.exists():
        raise FileNotFoundError(f"resume_from does not exist: {resume_from}")
    existing = json.loads(resume_from.read_text(encoding='utf-8'))
    reviews = existing.get('reviews', []) if isinstance(existing, dict) else existing
    for r in reviews:
        pid = r.get('proposal_id')
        ev = r.get('evaluator')
        if pid and ev:
            completed_keys.add((pid, ev))
    print(f"✓ Resuming from {resume_from} with {len(completed_keys)} completed (proposal_id, evaluator) pairs")


def save_reviews(path: Path, payload: Dict[str, Any]) -> None:
    path.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding='utf-8')


# Evaluate
save_every = 5
n_calls = 0

for evaluator in models_to_use:
    iterable = proposals_df.to_dict(orient='records')
    for proposal in tqdm(iterable, desc=f"Evaluating with {evaluator}"):
        key = (proposal['proposal_id'], evaluator)
        if key in completed_keys:
            continue

        prompt = make_prompt(proposal)
        raw = call_evaluator(prompt, evaluator)

        parsed, parse_err = extract_first_json_object(raw)

        entry = {
            'proposal_id': proposal['proposal_id'],
            'title': proposal['title'],
            'author': proposal['author'],
            'evaluator': evaluator,
            'evaluations': parsed if parsed is not None else None,
            'raw_response': raw,
            'parse_error': parse_err,
            'created_at': datetime.now().isoformat(),
        }

        reviews.append(entry)
        completed_keys.add(key)
        n_calls += 1

        if n_calls % save_every == 0:
            payload = {
                'created_at': datetime.now().isoformat(),
                'research_call_source': 'data/call_and_info.json',
                'human_inputs': [str(human_y1_path), str(human_y2_path)],
                'ai_input': str(ai_baseline_csv),
                'evaluators': models_to_use,
                'reviews': reviews,
            }
            save_reviews(output_path, payload)

# Final save
payload = {
    'created_at': datetime.now().isoformat(),
    'research_call_source': 'data/call_and_info.json',
    'human_inputs': [str(human_y1_path), str(human_y2_path)],
    'ai_input': str(ai_baseline_csv),
    'evaluators': models_to_use,
    'reviews': reviews,
}

save_reviews(output_path, payload)
print(f"\n✓ Saved consolidated reviews to: {output_path}")
print(f"  Total review entries: {len(reviews)}")

Evaluating with gemini-3-pro-preview:   0%|          | 0/92 [00:00<?, ?it/s]INFO:root:AFC is enabled with max remote calls: 10.
INFO:root:AFC remote call 1 is done.
Evaluating with gemini-3-pro-preview:   1%|          | 1/92 [00:27<41:18, 27.24s/it]INFO:root:AFC is enabled with max remote calls: 10.
INFO:root:AFC remote call 1 is done.
Evaluating with gemini-3-pro-preview:   2%|▏         | 2/92 [00:58<44:33, 29.71s/it]INFO:root:AFC is enabled with max remote calls: 10.
INFO:root:AFC remote call 1 is done.
Evaluating with gemini-3-pro-preview:   3%|▎         | 3/92 [01:26<42:45, 28.83s/it]INFO:root:AFC is enabled with max remote calls: 10.
INFO:root:AFC remote call 1 is done.
Evaluating with gemini-3-pro-preview:   4%|▍         | 4/92 [01:52<40:48, 27.83s/it]INFO:root:AFC is enabled with max remote calls: 10.
INFO:root:AFC remote call 1 is done.
Evaluating with gemini-3-pro-preview:   5%|▌         | 5/92 [02:17<38:32, 26.58s/it]INFO:root:AFC is enabled with max remote calls: 10.
INFO:ro


✓ Saved consolidated reviews to: data/reviews/ai_reviews/ai_reviews_ncems_criteria_20260223_153411.json
  Total review entries: 276
